In [1]:
import sys
import os

# Get the absolute path to the .src directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

In [2]:
DEVICE = 1
SEED = 42

In [3]:
import torch
from src.data import QuickDrawDataset

train_dataset = QuickDrawDataset("../../sketch_representations/data/quickdraw", split="train")
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=512, collate_fn=QuickDrawDataset.collate_fn_padd, 
                                           num_workers=8)

val_dataset = QuickDrawDataset("../../sketch_representations/data/quickdraw", split="valid")
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=512, collate_fn=QuickDrawDataset.collate_fn_padd, 
                                           num_workers=8)

test_dataset = QuickDrawDataset("../../sketch_representations/data/quickdraw", split="test")
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=512, collate_fn=QuickDrawDataset.collate_fn_padd, 
                                           num_workers=8)

In [4]:
import pickle as pkl
import torch
import numpy as np
from tqdm import tqdm
from src.lightning_models import LtSketchReconstruction
from src.models import Sketchformer, BlockConfig, PosEmbeddingConfig
from src.data import InputHandler, OutputHandler


for ckpt_path in tqdm(os.listdir("../../sketch_representations/checkpoints/experiment2/")):
    if not os.path.exists(f"../../sketch_representations/results/experiment2/{ckpt_path}.pkl"): continue

    with open(f"../../sketch_representations/results/experiment2/{ckpt_path}.pkl", "rb") as file:
        results = pkl.load(file)

    checkpoint = torch.load(f"../../sketch_representations/checkpoints/experiment2/{ckpt_path}/best.ckpt")

    sketchformer = Sketchformer(
        hidden_dim=results['args']['hidden_dim'],
        num_encoder_layers=results['args']['num_encoder_layers'],
        num_decoder_layers=results['args']['num_decoder_layers'],
        decoder_type=results['args']['decoder_type'],
        block_config=BlockConfig(
            dropout=results['args']['hidden_dropout']
        ),
        pos_embedding_config=PosEmbeddingConfig(
            pen_state=results['args']['pen_state'], 
            stroke_embedding=results['args']['stroke_embedding'], 
            sketch_pos=results['args']['sketch_pos'],
            stroke_pos=results['args']['stroke_pos']
        )
    )


    input_handler = InputHandler(input_relative_coords=results['args']['input_relative_coords'], output_relative_coords=results['args']['output_relative_coords'], autoencoder=True, autoregressive=results['args']['decoder_type'] in ["ar", "ar-enc"])
    output_handler = OutputHandler(output_relative_coords=results['args']['output_relative_coords'], autoregressive=results['args']['decoder_type'] in ["ar", "ar-enc"])
    model = LtSketchReconstruction(sketchformer, input_handler, output_handler, results['args']['lr'])
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    model = model.to("cuda:1")
    sketchformer = model.sketchformer
    sketchformer.decoder = None
    sketchformer.ln_decoder = None

    
    embeddings = {"train": ([], []), "val": ([], []), "test": ([], [])}

    for split, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
        for batch in loader:
            batch = {k: v.to("cuda:1") for k,v in batch.items()}
            model_input = input_handler(batch)
            x_enc = model_input['encoder']
            
            with torch.no_grad():
                h_sketch = sketchformer.encode(x_enc['pos'], x_enc['pos_info'], x_enc['token_id'], x_enc['mask'])

            embeddings[split][0].append(h_sketch.cpu())
            embeddings[split][1].append(batch['label'].cpu())

    for split in embeddings.keys():
        embeddings[split][0] = torch.cat(embeddings[split][0], dim=0)
        embeddings[split][1] = torch.cat(embeddings[split][1])

    embeddings['vars'] = results['vars']
    torch.save(embeddings, f"../embeddings/experiment2/{ckpt_path}.pt")

  0%|          | 0/27 [00:00<?, ?it/s]Exception ignored in: <function _releaseLock at 0x7f739e95f240>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/logging/__init__.py", line 237, in _releaseLock
    def _releaseLock():
    
KeyboardInterrupt: 
  0%|          | 0/27 [06:08<?, ?it/s]


RuntimeError: DataLoader worker (pid(s) 8667, 8678) exited unexpectedly